# TensorGated-SSM — Self-contained Reproducibility Notebook

A compact spectral–spatial state-space network for hyperspectral image classification.
This **single, independent** notebook (no external project files needed) reproduces the
**TensorGated-SSM (Ours)** result end-to-end on a chosen dataset:
PCA reduction → patch extraction → model training → evaluation (OA / AA / κ, per-class) →
prediction map → saved results.

**Model:** dual *distinct* tokenisation (spatial tokens from a CNN stem, spectral tokens from the raw
PCA cube) → center-conditioned feature-enhancement gate → one spectral–spatial cross-attention block →
a linear state-space scan → linear classifier (~62K parameters).

**Requirements:** `tensorflow==2.15`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`.
Put the dataset `.mat` files in `DATA_DIR` (default `./datasets`).

Repository: https://github.com/MHassaanButt/TensorGatedSSM


In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import tensorflow as tf
from scipy.io import loadmat
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             cohen_kappa_score, confusion_matrix)
from operator import truediv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from tensorflow.keras.layers import (Conv2D, BatchNormalization, Dense, Dropout,
                                     Activation, Reshape)
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g, True)
    except RuntimeError: pass
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ---------------- Configuration ----------------
DATA_DIR  = './datasets'   # folder containing the .mat files
DATASET   = 'HC'           # HC, HH, Qingyun, Tangdaowan, SA, PU
K         = 15             # PCA components
WS        = 14             # patch size (WS x WS)
TE_RATIO  = 0.90; TR_RATIO = 0.50; VR_RATIO = 0.50   # ~5% train / ~5% val / 90% test
EPOCHS    = 50; BATCH = 56; SEED = 345
OUT_DIR   = os.path.join('results', DATASET); os.makedirs(OUT_DIR, exist_ok=True)

# .mat file names / keys and class names for the six datasets used in the paper.
DATASETS = {
  'HC': dict(file='WHU_Hi_HanChuan.mat', key='WHU_Hi_HanChuan',
             gt_file='WHU_Hi_HanChuan_gt.mat', gt_key='WHU_Hi_HanChuan_gt', num_classes=16,
             names=['Strawberry','Cowpea','Soybean','Sorghum','Water spinach','Watermelon','Greens',
                    'Trees','Grass','Red roof','Gray roof','Plastic','Bare soil','Road','Bright object','Water']),
  'HH': dict(file='WHU_Hi_HongHu.mat', key='WHU_Hi_HongHu',
             gt_file='WHU_Hi_HongHu_gt.mat', gt_key='WHU_Hi_HongHu_gt', num_classes=22,
             names=['Red roof','Road','Bare soil','Cotton','Cotton firewood','Rape','Chinese cabbage','Pakchoi',
                    'Cabbage','Tuber mustard','Brassica parachinensis','Brassica chinensis','Small Brassica chinensis',
                    'Lactuca sativa','Celtuce','Film covered lettuce','Romaine lettuce','Carrot','White radish',
                    'Garlic sprout','Broad bean','Tree']),
  'Qingyun': dict(file='QUH-Qingyun.mat', key='Chengqu',
             gt_file='QUH-Qingyun_GT.mat', gt_key='ChengquGT', num_classes=6,
             names=['Trees','Concrete building','Car','Ironhide building','Plastic playground','Asphalt road']),
  'Tangdaowan': dict(file='QUH-Tangdaowan.mat', key='Tangdaowan',
             gt_file='QUH-Tangdaowan_GT.mat', gt_key='TangdaowanGT', num_classes=18,
             names=['Rubber track','Flaggingv','Sandy','Asphalt','Boardwalk','Rocky shallows','Grassland','Bulrush',
                    'Gravel road','Ligustrum vicaryi','Coniferous pine','Spiraea','Bare soil','Buxus sinica',
                    'Photinia serrulata','Populus','Ulmus pumila L','Seawater']),
  'SA': dict(file='Salinas_corrected.mat', key='salinas_corrected',
             gt_file='Salinas_gt.mat', gt_key='salinas_gt', num_classes=16,
             names=['Weeds_1','Weeds_2','Fallow','Fallow_rough_plow','Fallow_smooth','Stubble','Celery',
                    'Grapes_untrained','Soil_vinyard_develop','Corn_Weeds','Lettuce_4wk','Lettuce_5wk',
                    'Lettuce_6wk','Lettuce_7wk','Vinyard_untrained','Vinyard_trellis']),
  'PU': dict(file='PaviaU.mat', key='paviaU',
             gt_file='PaviaU_gt.mat', gt_key='paviaU_gt', num_classes=9,
             names=['Asphalt','Meadows','Gravel','Trees','Painted','Soil','Bitumen','Bricks','Shadows']),
}

In [ ]:
# ---------------- Data preparation ----------------
def load_hsi(name):
    cfg = DATASETS[name]
    HSI = loadmat(os.path.join(DATA_DIR, cfg['file']))[cfg['key']]
    GT  = loadmat(os.path.join(DATA_DIR, cfg['gt_file']))[cfg['gt_key']]
    return HSI, GT, cfg['num_classes'], cfg['names']

def pca_reduce(HSI, nc):
    X = HSI.reshape(-1, HSI.shape[2])
    X = PCA(n_components=nc, whiten=True).fit_transform(X)
    return X.reshape(HSI.shape[0], HSI.shape[1], nc)

def image_cubes(HSI, GT, WS):
    m = WS // 2
    pad = np.zeros((HSI.shape[0] + 2*m, HSI.shape[1] + 2*m, HSI.shape[2]))
    pad[m:m + HSI.shape[0], m:m + HSI.shape[1], :] = HSI
    data = np.zeros((HSI.shape[0]*HSI.shape[1], WS, WS, HSI.shape[2]))
    lab  = np.zeros(HSI.shape[0]*HSI.shape[1])
    idx = 0
    for r in range(m, pad.shape[0]-m):
        for c in range(m, pad.shape[1]-m):
            data[idx] = pad[r-m:r+m, c-m:c+m, :]
            lab[idx]  = GT[r-m, c-m]
            idx += 1
    keep = lab > 0
    return data[keep], (lab[keep] - 1).astype(int)

def tr_te_split(X, y, trR, vrR, teR, seed=345):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=teR, random_state=seed, stratify=y)
    Xtr, Xva, ytr, yva = train_test_split(Xtr, ytr, test_size=vrR/(trR+vrR),
                                          random_state=seed, stratify=ytr)
    return Xtr, Xva, Xte, ytr, yva, yte

In [ ]:
# ---------------- TensorGated-SSM model ----------------
class TokenGeneration(tf.keras.layers.Layer):
    '''Spatial tokens from CNN features; spectral tokens from the raw PCA cube.'''
    def __init__(self, d, **kw):
        super().__init__(**kw); self.spatial = Dense(d); self.spectral = Dense(d)
    def call(self, x_raw, x_cnn):
        B = tf.shape(x_raw)[0]; H, W = x_raw.shape[1], x_raw.shape[2]
        s = self.spatial(tf.reshape(x_cnn, [B, H*W, x_cnn.shape[3]]))
        p = self.spectral(tf.reshape(x_raw, [B, H*W, x_raw.shape[3]]))
        return s, p

class FeatureEnhancement(tf.keras.layers.Layer):
    '''Center-token conditioned sigmoid gating of both streams.'''
    def __init__(self, d, **kw):
        super().__init__(**kw)
        self.sg = Sequential([Dense(d), Activation('sigmoid'), Reshape((1, d))])
        self.pg = Sequential([Dense(d), Activation('sigmoid'), Reshape((1, d))])
    def call(self, s, p, center):
        return s * self.sg(center), p * self.pg(center)

class CrossAttention(tf.keras.layers.Layer):
    '''Multi-head cross-attention: spatial tokens query spectral tokens.'''
    def __init__(self, num_heads, d_model, dropout=0.1, **kw):
        super().__init__(**kw)
        self.h = num_heads; self.d = d_model; self.dh = d_model // num_heads
        self.wq = Dense(d_model); self.wk = Dense(d_model); self.wv = Dense(d_model)
        self.out = Dense(d_model); self.drop = Dropout(dropout)
    def _split(self, x, B):
        return tf.transpose(tf.reshape(x, (B, -1, self.h, self.dh)), [0, 2, 1, 3])
    def call(self, q, k, v, training=False):
        B = tf.shape(q)[0]; seq = q.shape[1] if q.shape[1] is not None else -1
        qp, kp, vp = self._split(self.wq(q), B), self._split(self.wk(k), B), self._split(self.wv(v), B)
        att = tf.nn.softmax(tf.matmul(qp, kp, transpose_b=True) / tf.math.sqrt(tf.cast(self.dh, tf.float32)), -1)
        o = tf.matmul(self.drop(att, training=training), vp)
        o = tf.reshape(tf.transpose(o, [0, 2, 1, 3]), [B, seq, self.d])
        return self.out(o)

class StateSpaceModel(tf.keras.layers.Layer):
    '''Linear recurrent scan h_t = W_h h_{t-1} + W_x x_t; returns the terminal state.'''
    def __init__(self, state_dim, **kw):
        super().__init__(**kw); self.sd = state_dim
        self.wh = Dense(state_dim); self.wx = Dense(state_dim)
    def call(self, x):
        st = tf.zeros([tf.shape(x)[0], self.sd])
        T = x.shape[1] if x.shape[1] is not None else x.shape.as_list()[1]
        for t in range(T):
            st = self.wh(st) + self.wx(x[:, t, :])
        return st

class TensorGatedSSM(tf.keras.Model):
    def __init__(self, out_channels=64, num_heads=4, state_dim=128, num_classes=16, dropout=0.1, **kw):
        super().__init__(**kw)
        self.pre = Sequential([Conv2D(16, 3, padding='same', activation='relu'), BatchNormalization(),
                               Conv2D(32, 3, padding='same', activation='relu'), BatchNormalization()])
        self.tok = TokenGeneration(out_channels)
        self.fe  = FeatureEnhancement(out_channels)
        self.attn = CrossAttention(num_heads, out_channels, dropout)
        self.ssm  = StateSpaceModel(state_dim)
        self.clf  = Dense(num_classes, activation='softmax')
    def call(self, x, training=False, return_features=False):
        xc = self.pre(x, training=training)
        s, p = self.tok(x, xc)
        center = tf.reduce_mean(s, axis=1, keepdims=True)
        s, p = self.fe(s, p, center)
        a = self.attn(s, p, p, training=training)
        h = self.ssm(a)
        return h if return_features else self.clf(h)

In [ ]:
# ---------------- Evaluation + prediction-map utilities ----------------
def classification_reports(TeC, pred, names):
    yt = np.argmax(TeC, 1) if TeC.ndim == 2 else TeC.astype(int)
    yp = np.argmax(pred, 1) if pred.ndim == 2 else pred.astype(int)
    rep = classification_report(yt, yp, target_names=names)
    cm  = confusion_matrix(yt, yp); oa = accuracy_score(yt, yp)
    each = np.nan_to_num(truediv(np.diag(cm), np.sum(cm, axis=1))); aa = np.mean(each)
    return rep, cm, oa*100, each*100, aa*100, cohen_kappa_score(yt, yp)*100

def predict_in_batches(m, X, chunk=16384):
    if len(X) <= chunk: return m.predict(X, verbose=0)
    return np.vstack([m.predict(X[i:i+chunk], verbose=0) for i in range(0, len(X), chunk)])

def gt_plot(all_patches, GT, model, batch_size=1024):
    preds = predict_in_batches(model, all_patches)
    predicted = np.argmax(preds, axis=1)
    H, W = GT.shape; out = np.zeros((H, W), dtype=int); n = 0
    for r in range(H):
        for c in range(W):
            if int(GT[r, c]) == 0: continue
            out[r, c] = predicted[n] + 1; n += 1
    return out

def save_prediction_map(out, num_classes, names, path, dpi=300):
    base = plt.cm.get_cmap('nipy_spectral')
    colors = np.vstack(([[0,0,0,1]], [base(i) for i in np.linspace(0.1, 0.9, num_classes)]))
    cmap = ListedColormap(colors)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(out, cmap=cmap, vmin=0, vmax=num_classes, interpolation='nearest'); ax.axis('off')
    fig.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0); plt.close(fig)

In [ ]:
# ---------------- Load data, reduce, patch, split ----------------
HSI, GT, num_classes, class_names = load_hsi(DATASET)
print(f'{DATASET}: HSI {HSI.shape}, {num_classes} classes')
RDHSI = pca_reduce(HSI, K); print('PCA ->', RDHSI.shape)
X, y  = image_cubes(RDHSI, GT, WS); print('patches', X.shape)
Xtr, Xva, Xte, ytr, yva, yte = tr_te_split(X, y, TR_RATIO, VR_RATIO, TE_RATIO, SEED)
print('train', Xtr.shape, '| val', Xva.shape, '| test', Xte.shape)

In [ ]:
# ---------------- Build + train ----------------
model = TensorGatedSSM(out_channels=64, num_heads=4, state_dim=128,
                       num_classes=num_classes, dropout=0.1)
_ = model(Xtr[:BATCH])
model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.legacy.Adam(lr=1e-3, decay=1e-6, clipnorm=1.0),
              metrics=['accuracy'])
print('Trainable parameters:', f'{model.count_params():,}')
t0 = time.time()
model.fit(Xtr, to_categorical(ytr), batch_size=BATCH, epochs=EPOCHS,
          validation_data=(Xva, to_categorical(yva)), verbose=2)
print(f'Training time: {time.time()-t0:.1f}s')

In [ ]:
# ---------------- Evaluate ----------------
pred = predict_in_batches(model, Xte)
rep, cm, oa, per_class, aa, kappa = classification_reports(to_categorical(yte), pred, class_names)
print(f'OA = {oa:.2f}%   AA = {aa:.2f}%   kappa = {kappa:.2f}%')
print(rep)

In [ ]:
# ---------------- Save results ----------------
result = dict(model='TensorGatedSSM', dataset=DATASET, seed=SEED,
              parameters=int(model.count_params()),
              OA=float(oa), AA=float(aa), Kappa=float(kappa),
              per_class_accuracy=[float(a) for a in per_class],
              class_names=list(class_names))
path = os.path.join(OUT_DIR, f'{DATASET}_TensorGatedSSM_seed{SEED}.json')
json.dump(result, open(path, 'w'), indent=2)
print('Saved', path)

In [ ]:
# ---------------- Full-scene prediction map ----------------
out = gt_plot(X, GT, model, batch_size=1024)
map_path = os.path.join(OUT_DIR, f'{DATASET}_TensorGatedSSM_PredMap.png')
save_prediction_map(out, num_classes, class_names, map_path)
plt.figure(figsize=(6, 6)); plt.imshow(plt.imread(map_path)); plt.axis('off'); plt.show()
print('Saved', map_path)

### Notes
* Paper numbers average over five seeds (`345, 445, 545, 645, 745`); loop `SEED` and average.
* Change `DATASET` to reproduce any of the six scenes with the identical protocol.
